In [ ]:
import requests
import time
import os
import json
from openpyxl import load_workbook  # pyright: ignore[reportMissingModuleSource]

# ==== CONFIG ====
BASE_URL = "http://localhost:8002/evaluate/rubric"
BASE_PATH = r"C:/Users/daves/Desktop/EduFlow/evaluation/downloaded_jobs"
EXCEL_PATH = r"C:\Users\daves\Desktop\EduFlow\evaluation\evaluation_results.xlsx"

JOB_IDS = [
    "job-1774534470161",
    "job-1774534999912",
    "job-1774544649734",
    "job-1774545286253",
    "job-1774545754239",
    "job-1774547498894",
    "job-1774548177351",
    "job-1774548651030",
    "job-1774549026730",
    "job-1774574045744",
    "job-1774574325545",
    "job-1774574626655",
    "job-1774590551422",
    "job-1774584736262",
    "job-1774584969330",
    "job-1774596210500",
    "job-1774595522117",
    "job-1774596683553",
]

CRITERIA = [
    "coherence",
    "dependency_flow",
    "content_progression",
    "non_redundancy"
]

RUNS = 3


# ==== LOAD EXCEL ====
wb = load_workbook(EXCEL_PATH)
ws = wb.active

# Find column indexes
header = [cell.value for cell in ws[1]]
job_id_col = header.index("Job ID") + 1
rubric_col = header.index("Rubric - LLM") + 1


def build_documents(job_id):
    pdf_folder = os.path.join(BASE_PATH, job_id, "pdf")
    return sorted([
        os.path.join(pdf_folder, f)
        for f in os.listdir(pdf_folder)
        if f.endswith(".pdf")
    ])


def update_excel(job_id, value):
    for row in ws.iter_rows(min_row=2):
        if row[job_id_col - 1].value == job_id:
            ws.cell(row=row[0].row, column=rubric_col, value=value)
            return True
    return False


# ==== MAIN LOOP ====
for idx, job_id in enumerate(JOB_IDS):
    print(f"\nProcessing {job_id} ({idx+1}/{len(JOB_IDS)})")

    documents = build_documents(job_id)

    payload = {
        "criteria": CRITERIA,
        "runs": RUNS,
        "sequence": {
            "architecture": "architecture A",
            "documents": documents
        }
    }

    try:
        response = requests.post(BASE_URL, json=payload)

        if response.status_code == 200:
            result = response.json()

            # Store full JSON response as string
            json_str = json.dumps(result)

            print(f"Response received")

            updated = update_excel(job_id, json_str)

            if updated:
                wb.save(EXCEL_PATH)
                print("Excel updated")
            else:
                print("Job ID not found in Excel")

        else:
            print(f"Failed: {response.status_code}")
            print(response.text)

    except Exception as e:
        print(f"Error: {str(e)}")

    # Wait 90 seconds (except last)
    if idx < len(JOB_IDS) - 1:
        print("Waiting 90 seconds...")
        time.sleep(90)

# Final save
wb.save(EXCEL_PATH)
print("\nDone!")


Processing job-1774547498894 (1/13)
Response received
Excel updated
Waiting 90 seconds...

Processing job-1774548177351 (2/13)
Response received
Excel updated
Waiting 90 seconds...

Processing job-1774548651030 (3/13)
Response received
Excel updated
Waiting 90 seconds...

Processing job-1774549026730 (4/13)
Response received
Excel updated
Waiting 90 seconds...

Processing job-1774574045744 (5/13)
Response received
Excel updated
Waiting 90 seconds...

Processing job-1774574325545 (6/13)
Response received
Excel updated
Waiting 90 seconds...

Processing job-1774574626655 (7/13)
Response received
Excel updated
Waiting 90 seconds...

Processing job-1774590551422 (8/13)
Response received
Excel updated
Waiting 90 seconds...

Processing job-1774584736262 (9/13)
Response received
Excel updated
Waiting 90 seconds...

Processing job-1774584969330 (10/13)
Response received
Excel updated
Waiting 90 seconds...

Processing job-1774596210500 (11/13)
Response received
Excel updated
Waiting 90 seconds.

In [ ]:
import requests
import time
import os
import json
from openpyxl import load_workbook  # pyright: ignore[reportMissingModuleSource]

# ==== CONFIG ====
BASE_URL = "http://localhost:8002/evaluate/pairwise"
BASE_PATH = r"C:/Users/daves/Desktop/EduFlow/evaluation/downloaded_jobs"
EXCEL_PATH = r"C:\Users\daves\Desktop\EduFlow\evaluation\evaluation_results.xlsx"

# 6 sets — each set is (EduFlow, Baseline1, Baseline2)
JOB_SETS = [
    ("job-1774534470161", "job-1774534999912", "job-1774544649734"),
    ("job-1774545286253", "job-1774545754239", "job-1774547498894"),
    ("job-1774548177351", "job-1774548651030", "job-1774549026730"),
    ("job-1774574045744", "job-1774574325545", "job-1774574626655"),
    ("job-1774590551422", "job-1774584736262", "job-1774584969330"),
    ("job-1774596210500", "job-1774595522117", "job-1774596683553"),
]

CRITERIA = [
    "coherence",
    "dependency_flow",
    "content_progression",
    "non_redundancy"
]

RUNS = 3


# ==== LOAD EXCEL ====
wb = load_workbook(EXCEL_PATH)
ws = wb.active

header = [cell.value for cell in ws[1]]
job_id_col = header.index("Job ID") + 1
pairwise_col = header.index("Pairwise - LLM") + 1


def build_documents(job_id):
    pdf_folder = os.path.join(BASE_PATH, job_id, "pdf")
    return sorted([
        os.path.join(pdf_folder, f)
        for f in os.listdir(pdf_folder)
        if f.endswith(".pdf")
    ])


def update_excel(job_id, value):
    for row in ws.iter_rows(min_row=2):
        if row[job_id_col - 1].value == job_id:
            ws.cell(row=row[0].row, column=pairwise_col, value=value)
            return True
    return False


# ==== MAIN LOOP ====
total_requests = len(JOB_SETS) * 2  # 2 comparisons per set
request_count = 0

for set_idx, (eduflow_id, baseline1_id, baseline2_id) in enumerate(JOB_SETS):
    print(f"\n{'='*50}")
    print(f"Processing set {set_idx + 1}/6")
    print(f"  EduFlow   : {eduflow_id}")
    print(f"  Baseline 1: {baseline1_id}")
    print(f"  Baseline 2: {baseline2_id}")
    print(f"{'='*50}")

    eduflow_docs = build_documents(eduflow_id)

    # ---- Comparison 1: EduFlow vs Baseline 1 ----
    request_count += 1
    print(f"\n[{request_count}/{total_requests}] EduFlow vs Baseline 1 ({baseline1_id})")

    baseline1_docs = build_documents(baseline1_id)

    payload = {
        "sequence_a": {
            "architecture": "EduFlow",
            "documents": eduflow_docs
        },
        "sequence_b": {
            "architecture": "Baseline 1",
            "documents": baseline1_docs
        },
        "criteria": CRITERIA,
        "runs": RUNS
    }

    try:
        response = requests.post(BASE_URL, json=payload)

        if response.status_code == 200:
            result = response.json()
            json_str = json.dumps(result)
            updated = update_excel(baseline1_id, json_str)
            if updated:
                wb.save(EXCEL_PATH)
                print(f"Saved to row: {baseline1_id}")
            else:
                print(f"Job ID not found in Excel: {baseline1_id}")
        else:
            print(f"Failed: {response.status_code}")
            print(response.text)

    except Exception as e:
        print(f"Error: {str(e)}")

    print("Sleeping 90 seconds...")
    time.sleep(90)

    # ---- Comparison 2: EduFlow vs Baseline 2 ----
    request_count += 1
    print(f"\n[{request_count}/{total_requests}] EduFlow vs Baseline 2 ({baseline2_id})")

    baseline2_docs = build_documents(baseline2_id)

    payload = {
        "sequence_a": {
            "architecture": "EduFlow",
            "documents": eduflow_docs
        },
        "sequence_b": {
            "architecture": "Baseline 2",
            "documents": baseline2_docs
        },
        "criteria": CRITERIA,
        "runs": RUNS
    }

    try:
        response = requests.post(BASE_URL, json=payload)

        if response.status_code == 200:
            result = response.json()
            json_str = json.dumps(result)
            updated = update_excel(baseline2_id, json_str)
            if updated:
                wb.save(EXCEL_PATH)
                print(f"Saved to row: {baseline2_id}")
            else:
                print(f"Job ID not found in Excel: {baseline2_id}")
        else:
            print(f"Failed: {response.status_code}")
            print(response.text)

    except Exception as e:
        print(f"Error: {str(e)}")

    # Sleep after every comparison except the very last one
    if set_idx < len(JOB_SETS) - 1:
        print("Sleeping 90 seconds...")
        time.sleep(90)

# Final save
wb.save(EXCEL_PATH)
print("\Done! All pairwise comparisons complete.")


Processing set 1/6
  EduFlow   : job-1774534470161
  Baseline 1: job-1774534999912
  Baseline 2: job-1774544649734

[1/12] EduFlow vs Baseline 1 (job-1774534999912)
Saved to row: job-1774534999912
Sleeping 90 seconds...

[2/12] EduFlow vs Baseline 2 (job-1774544649734)
Saved to row: job-1774544649734
Sleeping 90 seconds...

Processing set 2/6
  EduFlow   : job-1774545286253
  Baseline 1: job-1774545754239
  Baseline 2: job-1774547498894

[3/12] EduFlow vs Baseline 1 (job-1774545754239)
Saved to row: job-1774545754239
Sleeping 90 seconds...

[4/12] EduFlow vs Baseline 2 (job-1774547498894)
Saved to row: job-1774547498894
Sleeping 90 seconds...

Processing set 3/6
  EduFlow   : job-1774548177351
  Baseline 1: job-1774548651030
  Baseline 2: job-1774549026730

[5/12] EduFlow vs Baseline 1 (job-1774548651030)
Saved to row: job-1774548651030
Sleeping 90 seconds...

[6/12] EduFlow vs Baseline 2 (job-1774549026730)
Saved to row: job-1774549026730
Sleeping 90 seconds...

Processing set 4/6
  